In [ ]:
from import_ipynb import NotebookFinder  # type: ignore
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score
import importlib
import os
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np
import json


In [ ]:
# retrouver les dossiers
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
utility_dir = os.path.join(root, "pneumonia_knn", "notebooks", "utility")
result_dir = os.path.join(root, "pneumonia_knn", "result")

# --- pneumonia_knn\notebooks\utility
spec_print = NotebookFinder().find_spec("prints", [utility_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

In [ ]:
# 1. Définir les métriques pour GridSearchCV
scoring = {
    'accuracy':  'accuracy',
    'precision': make_scorer(precision_score, average='weighted'),
    'recall':    make_scorer(recall_score,    average='macro'),
    'f1':        make_scorer(f1_score,        average='weighted'),
    'roc_auc':   make_scorer(roc_auc_score,   response_method="predict_proba", average='macro', multi_class='ovo')
}
def get_scoring():
    return scoring
# 2. Colonnes à afficher dans les résultats
cols = [
    'Index (ordre) CV',
    'mean_test_accuracy',
    'mean_test_precision',
    'mean_test_recall',
    'mean_test_f1',
    'mean_test_roc_auc',
    'rank_test_accuracy'
]

### Visualiser les composantes principales du PCA
Cette fonction affiche les premières composantes principales sous forme d'images.

In [ ]:
def plot_pca_components(pca):
    fig, axes = plt.subplots(1, 5, figsize=(14, 3))

    for i, ax in enumerate(axes):
        composante = pca.components_[i].reshape(224, 224, 3)  # ← 3 canaux
        composante = (composante - composante.min()) / (composante.max() - composante.min())
        
        ax.imshow(composante)  # ← pas cmap='gray'
        ax.set_title(f'PC{i+1} — {pca.explained_variance_ratio_[i]*100:.1f}%', fontsize=9)
        ax.axis('off')

    plt.suptitle("Ce que chaque composante PCA a capturé", fontsize=11)
    plt.tight_layout()
    plt.show()

### Visualiser la variance expliquée par le PCA
Ce graphique montre la variance expliquée et la variance cumulée par composante principale.

In [ ]:
def plot_pca_variance_explained(pca):
    variance_expliquee = pca.explained_variance_ratio_ * 100
    variance_cumulee = np.cumsum(variance_expliquee)
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Barres - variance par composante
    ax1.bar(range(1, len(variance_expliquee) + 1), variance_expliquee, color='steelblue', alpha=0.7, label='Variance expliquée par composante')
    ax1.set_xlabel('Composantes principales')
    ax1.set_ylabel('Variance expliquée (%)', color='steelblue')

    # Courbe - variance cumulée
    ax2 = ax1.twinx()
    ax2.plot(range(1, len(variance_cumulee) + 1), variance_cumulee, color='red', marker='o', markersize=0.5, label='Variance cumulée')
    ax2.set_ylabel('Variance cumulée (%)', color='red')
    ax2.axhline(y=95, color='gray', linestyle='--', label='Seuil 95%')

    plt.title('Scree Plot — Variance expliquée par le PCA')
    fig.legend(loc='center right')
    plt.tight_layout()
    plt.show()

### Afficher le résultat Cross-Validation de GridSearchCV


In [ ]:
def cross_validation_png(grid, implementation_name):
    results = pd.DataFrame(grid.cv_results_)  # ← ajoute cette ligne
    # Sélectionner les colonnes clés
    table_df = results[[
        'params',
        'mean_test_accuracy',
        'mean_test_precision',
        'mean_test_recall',
        'mean_test_f1',
        'mean_test_roc_auc',
        'rank_test_accuracy'
    ]].copy()
    table_df.insert(0, 'Index CV', [f'#{i+1}' for i in range(len(table_df))])  # ← ici

    # Afficher seulement l'index dans la colonne params
    table_df['params'] = [str(i+1) for i in range(len(table_df))]
    
    # Supprimer la colonne params (redondante avec Index CV)
    table_df = table_df.drop(columns=['params'])

    # Ajuster la taille en fonction du nombre de lignes et colonnes
    num_cols = len(table_df.columns)
    num_rows = len(table_df)
    fig_width = 2 + num_cols * 1.8  # largeur adaptée au nombre de colonnes
    fig_height = 1.5 + num_rows * 0.45  # hauteur adaptée au nombre de lignes
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc='center',
        loc='center',
        colColours=['#f7f7f7'] + ['#ffffff'] * (num_cols - 1)
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)

    # Trouver l'indice de la ligne avec rank_test_accuracy = 1
    rank_col_idx = table_df.columns.get_loc('rank_test_accuracy')
    best_rank_row = None
    for i in range(num_rows):
        if table_df.iloc[i, rank_col_idx] == 1:
            best_rank_row = i + 1  # +1 car la ligne 0 est l'en-tête
            break

    # Styliser l'en-tête
    for j in range(num_cols):
        table[(0, j)].set_facecolor('#40466e')
        table[(0, j)].set_text_props(weight='bold', color='white')

    # Alternance de couleur sur les lignes + surbrillance du meilleur rang
    for i in range(1, num_rows + 1):
        for j in range(num_cols):
            if i == best_rank_row:
                # Surbrillance pour le meilleur rang
                table[(i, j)].set_facecolor('#FFD700')  # Doré
                table[(i, j)].set_text_props(weight='bold')
            elif i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')

    plt.title(implementation_name + " Cross-Validation Results", fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    output_path = f'{result_dir}/fine_tuning/cross_validation_{implementation_name}.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.show()

In [ ]:
def cross_validation_json(grid, implementation_name):
    # Préparer les données à sauvegarder
    results_data = {
        'implementation': implementation_name,
        'best_params': grid.best_params_,
        'best_score': float(grid.best_score_),
        'best_index': int(grid.best_index_),
        'cv_results': {}
    }
    
    # Convertir les colonnes numpy en listes pour la sérialisation JSON
    for key, value in grid.cv_results_.items():
        if hasattr(value, 'tolist'):
            results_data['cv_results'][key] = value.tolist()
        else:
            results_data['cv_results'][key] = value
    
    # Sauvegarder en JSON
    output_path = f'{result_dir}/fine_tuning/cross_validation_{implementation_name}.json'
    with open(output_path, 'w') as f:
        json.dump(results_data, f, indent=4)

### Afficher le tableau GridSearchCV complet
Affiche les colonnes d'intérêt triées par rang et sauvegarde une image du tableau.

In [ ]:
def display_grid_search_results_table(results, title):
    print(title)
    display(results[cols].sort_values('rank_test_accuracy'))

### Afficher les résultats triés de GridSearchCV
Présente un tableau des meilleures configurations selon le rang d'accuracy.

In [ ]:
def plot_knn_prediction_scatter(y_test, y_pred, X_reduced, implementation_name):
    class_names = ['NORMAL', 'BACTERIA', 'VIRUS']
    colors      = ['#378ADD', '#22a05a', '#EF9F27']
    
    correct = (y_test == y_pred)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Projection {implementation_name} — KNN predictions", fontsize=14)
    
    # --- Onglet 1 : Vraie classe ---
    ax = axes[0]
    for c, (name, color) in enumerate(zip(class_names, colors)):
        mask = y_test == c
        ax.scatter(X_reduced[mask, 0], X_reduced[mask, 1],
                   c=color, label=name, alpha=0.6, s=15)
    ax.set_title("Vraie classe")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    # --- Onglet 2 : Prédiction KNN ---
    ax = axes[1]
    for c, (name, color) in enumerate(zip(class_names, colors)):
        mask = y_pred == c
        ax.scatter(X_reduced[mask, 0], X_reduced[mask, 1],
                   c=color, label=f"Prédit {name}", alpha=0.6, s=15)
    ax.set_title("Prédiction KNN")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    # --- Onglet 3 : Erreurs ---
    ax = axes[2]
    ax.scatter(X_reduced[correct, 0],  X_reduced[correct, 1],
               c='#cccccc', alpha=0.3, s=12, label='Correct')
    ax.scatter(X_reduced[~correct, 0], X_reduced[~correct, 1],
               c='#e24b4a', alpha=0.8, s=20, label='Erreur')
    ax.set_title("Erreurs de prédiction")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

### Visualiser les prédictions KNN
Affiche la vraie classe, les prédictions et les erreurs dans des nuages de points.

In [ ]:
def plot_prediction_metrics_bar(y_test, y_pred, title = "Métriques de prédiction", y_proba = None):
    metrics = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall':    recall_score(y_test, y_pred, average='weighted'),
        'F1':        f1_score(y_test, y_pred, average='weighted'),
    }
    
    if y_proba is not None:
        metrics['ROC AUC'] = roc_auc_score(y_test, y_proba, average='macro', multi_class='ovo')

    fig, ax = plt.subplots(figsize=(14, 6))
    
    bars = ax.barh(
        list(metrics.keys()), 
        list(metrics.values()), 
        color=['#378ADD', '#22a05a', '#EF9F27', '#e24b4a', '#9b59b6'],
        alpha=0.8
    )    
    for bar, val in zip(bars, metrics.values()):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)
    
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Score", fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='both', labelsize=8)
    ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Visualiser les scores de prédiction
Affiche un histogramme des métriques globales du modèle KNN.

In [ ]:
def plot_true_positives_by_class(y_test, y_pred, title=""):
    classes = ['NORMAL', 'BACTERIA', 'VIRUS']
    cm = confusion_matrix(y_test, y_pred)
    
    # Les vrais positifs sont sur la diagonale
    tp = cm.diagonal()
    total = cm.sum(axis=1)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    x = np.arange(len(classes))
    bars_tp    = ax.bar(x - 0.2, tp,    width=0.4, label='Vrais positifs', color='#22a05a', alpha=0.8)
    bars_total = ax.bar(x + 0.2, total, width=0.4, label='Total réel',     color='#378ADD', alpha=0.8)
    
    # Afficher les valeurs sur les barres
    for bar in bars_tp:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', fontsize=9)
    for bar in bars_total:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', fontsize=9)
    
    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=10)
    ax.set_ylabel("Nombre d'images", fontsize=9)
    ax.set_title(f"Vrais positifs par classe — {title}", fontsize=11)
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()


### Visualiser les vrais positifs par classe
Compare les vrais positifs et les totaux réels pour chaque classe.